In [ ]:
# ===== Step 7 - label conflicts, leakage audit, text the image cannot support =====
# STANDALONE: attach the output of the "IE643 Cleaning" notebook (Version 2) as an input.
# Reads its reports_clean.csv / images_512.npy in place, never drops or reorders rows
# (row i stays aligned with images_512.npy[i] of IE643 Cleaning) and writes the new CSV to /kaggle/working.
#   7a  No-Finding label conflicts  -> y_no_finding = 0 for rows with a positive pathology
#   7b  Text rules                  -> impression_final (= impression_clean minus view/other-exam/recommendation text)
#   7c  Audit                       -> duplicate texts, duplicate images, patient overlap across splits
# impression_clean and impression_noprior are left exactly as they were.
import re, json, hashlib, collections
import numpy as np, pandas as pd

import glob, os
W = "/kaggle/working"
hits = glob.glob("/kaggle/input/**/reports_clean.csv", recursive=True)
assert len(hits) == 1, f"expected exactly one reports_clean.csv under /kaggle/input, found {hits}"
SRC = os.path.dirname(hits[0])
CSV, NPY = f"{SRC}/reports_clean.csv", f"{SRC}/images_512.npy"
OUT = f"{W}/reports_clean.csv"
print("Reading from", SRC)
CAP = 160

df = pd.read_csv(CSV)
for c in ["impression_raw", "impression_clean", "impression_noprior"]:
    df[c] = df[c].fillna("").astype(str)
imgs = np.load(NPY, mmap_mode="r")
assert len(df) == imgs.shape[0], (len(df), imgs.shape)
assert (df["idx"].values == np.arange(len(df))).all(), "idx is not 0..N-1 in order"
S7 = {"rows": int(len(df))}

# ---------------------------------------------------------------- 7a labels
print("=== 7a  No Finding conflicts ===")
PATH = ["Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
        "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture"]  # Support Devices excluded by CheXpert's own rule
nf = df["No Finding"].eq(1)
pos = df[PATH].eq(1)
unc = df[PATH].eq(-1)
conf = nf & pos.any(axis=1)
rev = (~nf) & (~pos.any(axis=1)) & (~unc.any(axis=1))
print(f"No Finding = 1: {int(nf.sum())} | with a positive pathology (conflict): {int(conf.sum())} | with an uncertain pathology: {int((nf & unc.any(axis=1)).sum())}")
print(f"Reverse case (No Finding != 1 but no positive/uncertain pathology): {int(rev.sum())}")
for i in df.index[conf]:
    r = df.loc[i]
    print(f"  idx {i:5d} [{r['subset']}] positives={[p for p in PATH if r[p] == 1]} | {r['impression_clean'][:260]}")
n_y0 = int((df.loc[conf, "y_no_finding"] == 1).sum())
df["label_conflict"] = conf
df["label_reverse_conflict"] = rev
df.loc[conf, "y_no_finding"] = 0
print(f"y_no_finding set to 0 for {n_y0} rows (flag: label_conflict). Reverse-case rows are only flagged (label_reverse_conflict).")
for i in df.index[rev][:8]:
    print(f"  reverse idx {i:5d} NF={df.loc[i, 'No Finding']} | {df.loc[i, 'impression_clean'][:200]}")
df.loc[conf, ["idx", "subset", "No Finding"] + PATH + ["impression_clean"]].to_csv(f"{W}/label_conflicts.csv", index=False)
S7["label_conflicts_fixed"] = n_y0
S7["label_reverse_conflicts_flagged"] = int(rev.sum())

# ---------------------------------------------------------------- 7b text rules
print("\n=== 7b  Text the image cannot support ===")
def split_sents(t):
    t = re.sub(r"\s+", " ", t).strip()
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if s.strip()]

VW = (r"(?:single|two|three|four|2|3|limited|portable|semi-?\s?upright|upright|supine|erect|recumbent|ap|pa|frontal|"
      r"lateral|oblique|decubitus|and|or|views?|projections?|films?|radiographs?|radiography|x-?rays?|chest|images?|of|the|"
      r"a|an|one|study|exam|examination|lat|ap/pa|pa/lat(?:eral)?|ap/lateral)")
VIEW_KEY = re.compile(r"\b(?:views?|projections?|films?|radiographs?|radiography|x-?rays?|images?|exam|examination|study)\b")
VIEW_LEAD = re.compile(rf"^((?:{VW}[\s,/-]+)+?)(?:demonstrates?|demonstrating|shows?|showing|reveals?|revealing|depicts?|displays?)\s+(?:that\s+)?", re.I)
VIEW_TOK = re.compile(rf"^{VW}$")

OTHER = re.compile(r"\b(?:shoulders?|humer\w+|elbows?|wrists?|forearms?|hands?|fingers?|thumbs?|hips?|femur\w*|knees?|ankles?|"
                   r"foot|feet|toes?|tibia\w*|fibula\w*|pelvi[sc]\w*|lumbar|sacr\w+|skull|mandible|facial)\b")
CHEST = re.compile(r"\b(?:lungs?|pulmonary|pleura\w*|pneumo\w*|cardi\w+|heart|mediastin\w*|hil(?:a|ar|um)|chest|thora\w+|ribs?|"
                   r"clavic\w+|trache\w+|bronch\w+|\w*diaphragm\w*|costophrenic|sulcus|tubes?|lines?|catheters?|picc|pacemaker|icd|"
                   r"stent|valve|stern\w+|aort\w+|svc|atri\w+|vein|port|drain|wires?|effusions?|atelecta\w+|edema|consolidat\w+|"
                   r"opaci\w+|nodul\w+|masses|mass|infiltrat\w*|emphysema|airspace|vascular\w*|spine|scoliosis)\b")
EXAM_CUE = re.compile(r"\b(?:views?|radiographs?|x-?rays?|series|exam|examination|study|imaging|films?)\b")

REC_TAIL = re.compile(
    r"(?:[,;:]\s*|\s+)(?:and\s+|but\s+|however,?\s+|also\s+)?(?:"
    r"if\s+clinically\s+(?:indicated|warranted|needed|appropriate)[^.]*"
    r"|(?:further|follow[- ]?up|short[- ]?term|repeat|dedicated|additional|continued)\b[^.]*?\b"
    r"(?:recommended|advised|suggested|warranted|indicated|obtained|performed|needed|helpful|useful|considered)\b[^.]*"
    r"|(?:recommend\w*|consider|consideration|clinical\s+correlation|correlate|correlation)\b[^.]*"
    r"|suggest\s+(?:follow|correlat|further|dedicated|ct|repeat)[^.]*"
    r")\.?$", re.I)

def words(s):
    return len(re.findall(r"[a-z]{3,}", s))

def finish(s):
    s = re.sub(r"[\s,;:-]+(?:and|or|but|with|if|which|that|to)$", "", s.strip()).strip(" ,;:-")
    return s + "." if s and s[-1] not in ".!?" else s

def apply_rules(text):
    out, ev, after_other = [], [], False
    for s in split_sents(text):
        orig = s
        if after_other:                       # short orphan left behind by a dropped other-exam sentence
            after_other = False
            if not CHEST.search(s) and not OTHER.search(s) and words(s) <= 7:
                ev.append(("other_exam", orig, "")); continue
        toks = re.findall(r"[a-z/-]+", s)
        if toks and VIEW_KEY.search(s) and all(VIEW_TOK.match(t) for t in toks):
            ev.append(("view_only", orig, "")); continue
        m = VIEW_LEAD.match(s)
        if m and VIEW_KEY.search(m.group(1)):
            s = s[m.end():]
            ev.append(("view_lead", orig, s))
            if words(s) < 2:
                continue
        if OTHER.search(s) and not CHEST.search(s) and EXAM_CUE.search(s):
            ev.append(("other_exam", orig, "")); after_other = True; continue
        m = REC_TAIL.search(" " + s)
        if m:
            kept = finish((" " + s)[:m.start()])
            ev.append(("recommendation", orig, kept))
            if words(kept) < 2:
                continue
            s = kept
        out.append(s)
    return " ".join(out).strip(), ev

res = [apply_rules(t) for t in df["impression_clean"]]
df["impression_final"] = [r[0] for r in res]
events = collections.defaultdict(list)
for i, (_, ev) in enumerate(res):
    for rule, b, a in ev:
        events[rule].append((i, b, a))
for rule in ["view_only", "view_lead", "other_exam", "recommendation"]:
    e = events[rule]
    n_rep = len({i for i, _, _ in e})
    print(f"\n[{rule}] {len(e)} sentences in {n_rep} reports ({n_rep / len(df) * 100:.1f}%)")
    rs = np.random.RandomState(0)
    for k in rs.permutation(len(e))[:6]:
        i, b, a = e[k]
        print(f"   idx {i}\n     before: {b[:230]}\n     after : {a[:230] or '<dropped>'}")
    S7[f"rule_{rule}"] = {"sentences": len(e), "reports": n_rep}

# kept-but-suspicious text, reported not edited
lat_left = int(df["impression_final"].str.contains(r"\blateral\b").sum())
other_kept = int(df["impression_final"].map(lambda t: any(OTHER.search(s) for s in split_sents(t))).sum())
print(f"\nReports still containing 'lateral': {lat_left} | reports with another-body-part word kept (no exam cue): {other_kept}")
S7["still_mention_lateral"], S7["other_body_part_kept"] = lat_left, other_kept

df["changed_by_step7"] = df["impression_final"].ne(df["impression_clean"])
df["final_empty"] = df["impression_final"].eq("")
df["use_for_training"] = ~df["final_empty"]
print(f"\nimpression_final differs from impression_clean in {int(df.changed_by_step7.sum())} reports "
      f"({df.changed_by_step7.mean() * 100:.1f}%); emptied entirely (flag final_empty, kept in file): {int(df.final_empty.sum())}")
for i in df.index[df.final_empty][:10]:
    print(f"   emptied idx {i}: {df.loc[i, 'impression_clean'][:220]}")
S7["changed_reports"], S7["emptied_reports"] = int(df.changed_by_step7.sum()), int(df.final_empty.sum())
pd.DataFrame([(r, i, b, a) for r, e in events.items() for i, b, a in e],
             columns=["rule", "idx", "before", "after"]).to_csv(f"{W}/step7_text_changes.csv", index=False)

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilgpt2")
def ntok(col):
    return np.array([len(x) for x in tok(df[col].tolist())["input_ids"]])
chk = ntok("impression_clean")
print(f"\nTokenizer sanity: recomputed ntok_clean equals stored column in {(chk == df['ntok_clean'].values).mean() * 100:.1f}% of rows")
df["ntok_final"] = ntok("impression_final")
df["over_cap_final"] = df["ntok_final"] > CAP
for name, a in [("clean", df["ntok_clean"].values), ("final", df["ntok_final"].values)]:
    print(f"tokens {name:5s}: p50 {np.percentile(a, 50):.0f} p90 {np.percentile(a, 90):.0f} p99 {np.percentile(a, 99):.0f} | >{CAP}: {(a > CAP).mean() * 100:.2f}%")
S7["tokens_final"] = {"p50": float(np.percentile(df.ntok_final, 50)), "p90": float(np.percentile(df.ntok_final, 90)),
                      "p99": float(np.percentile(df.ntok_final, 99)), "pct_over_cap": float(df.over_cap_final.mean() * 100)}

# ---------------------------------------------------------------- 7c audit
print("\n=== 7c  Audit ===")
tr = set(df.loc[df.subset == "train", "deid_patient_id"]); va = set(df.loc[df.subset == "val", "deid_patient_id"]); te = set(df.loc[df.subset == "test", "deid_patient_id"])
ov = {"train&val": len(tr & va), "train&test": len(tr & te), "val&test": len(va & te)}
print("Patients shared between splits:", ov)
assert sum(ov.values()) == 0, "PATIENT LEAKAGE ACROSS SPLITS"
S7["patient_overlap"] = ov

for col in ["impression_clean", "impression_final"]:
    vc = df.loc[df.use_for_training, col].value_counts()
    dup_rows = int((df.loc[df.use_for_training, col].map(vc) > 1).sum())
    trs = set(df.loc[(df.subset == "train") & df.use_for_training, col])
    te_rows = df[(df.subset == "test") & df.use_for_training]
    share = te_rows[col].isin(trs).mean() * 100
    print(f"{col}: {vc.size} distinct texts for {int(df.use_for_training.sum())} rows | rows sharing a text with another row: {dup_rows} "
          f"({dup_rows / df.use_for_training.sum() * 100:.1f}%) | test reports with an exact copy in train: {share:.1f}%")
    S7[f"dup_{col}"] = {"distinct": int(vc.size), "rows_in_duplicates": dup_rows, "test_exact_in_train_pct": float(share)}
    if col == "impression_final":
        print("   most repeated:", [(t[:70], int(n)) for t, n in vc.head(6).items()])

hashes = collections.defaultdict(list)
for i in range(len(imgs)):
    hashes[hashlib.md5(np.ascontiguousarray(imgs[i]).tobytes()).hexdigest()].append(i)
groups = [g for g in hashes.values() if len(g) > 1]
cross = [g for g in groups if df.loc[g, "subset"].nunique() > 1]
same_pat = [g for g in groups if df.loc[g, "deid_patient_id"].nunique() == 1]
print(f"Identical 512x512 images: {len(groups)} groups ({sum(len(g) for g in groups)} images) | spanning splits: {len(cross)} | same patient: {len(same_pat)}")
for g in groups[:5]:
    print("   idx", g, "subsets", df.loc[g, "subset"].tolist(), "patients", df.loc[g, "deid_patient_id"].tolist())
S7["duplicate_image_groups"] = {"groups": len(groups), "spanning_splits": len(cross), "same_patient": len(same_pat)}
df["dup_image_group"] = -1
for gi, g in enumerate(groups):
    df.loc[g, "dup_image_group"] = gi

# ---------------------------------------------------------------- write
assert (df["idx"].values == np.arange(len(df))).all() and len(df) == imgs.shape[0]
df.to_csv(OUT, index=False)
summ = json.load(open(f"{SRC}/cleaning_summary.json"))
summ["step7"] = S7
summ["text"]["training_target"] = "impression_final (impression_clean + Step 7 text rules); impression_noprior kept for a later ablation"
json.dump(summ, open(f"{W}/cleaning_summary.json", "w"), indent=2)
json.dump(S7, open(f"{W}/step7_report.json", "w"), indent=2)
print("\nStep 7 done. New columns: impression_final, ntok_final, over_cap_final, final_empty, use_for_training, changed_by_step7, "
      "label_conflict, label_reverse_conflict, dup_image_group.")